# Cat vs Dog CNN (CIFAR-10) — Colab version
Features: dataset loaded from Google Drive, per-epoch loss/accuracy graph (saved to Drive), and checkpoints saved to Drive every epoch so training **resumes automatically** if the session or network drops.

**Before running:** set `Runtime > Change runtime type > GPU`.

## 1. Mount Google Drive and set paths

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os

# ---- EDIT THESE ----
# Where your CIFAR-10 data lives on Drive. Either:
#   (a) the extracted folder  "cifar-10-batches-py"  inside DRIVE_DATA_DIR, or
#   (b) the archive           "cifar-10-python.tar.gz" inside DRIVE_DATA_DIR
DRIVE_DATA_DIR = "/content/drive/MyDrive/datasets/cifar10"

# Where checkpoints, plots and the final model get saved on Drive
PROJECT_DIR = "/content/drive/MyDrive/cat_dog_cnn"

EPOCHS = 20
BATCH_SIZE = 64
LEARNING_RATE = 0.001
# --------------------

CKPT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)

LATEST_CKPT = os.path.join(CKPT_DIR, "latest.pth")      # overwritten every epoch (for resuming)
BEST_CKPT   = os.path.join(CKPT_DIR, "best.pth")        # best test accuracy so far
PLOT_PATH   = os.path.join(PROJECT_DIR, "training_curves.png")
HISTORY_PATH = os.path.join(PROJECT_DIR, "history.json")
FINAL_MODEL = os.path.join(PROJECT_DIR, "cat_dog_model.pth")

print("Project folder:", PROJECT_DIR)

## 2. Imports and device

In [ ]:
import json, shutil, tarfile, time
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from IPython.display import clear_output

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

## 3. Load CIFAR-10 from Drive
The data is copied/extracted to Colab's local disk first (`/content/data`), because reading thousands of files straight from Drive is slow.

In [ ]:
LOCAL_ROOT = "/content/data"
LOCAL_BATCHES = os.path.join(LOCAL_ROOT, "cifar-10-batches-py")
os.makedirs(LOCAL_ROOT, exist_ok=True)

drive_folder  = os.path.join(DRIVE_DATA_DIR, "cifar-10-batches-py")
drive_archive = os.path.join(DRIVE_DATA_DIR, "cifar-10-python.tar.gz")

if os.path.isdir(LOCAL_BATCHES):
    print("Data already on local disk.")
elif os.path.isdir(drive_folder):
    print("Copying extracted folder from Drive...")
    shutil.copytree(drive_folder, LOCAL_BATCHES)
elif os.path.isfile(drive_archive):
    print("Extracting archive from Drive...")
    with tarfile.open(drive_archive, "r:gz") as tar:
        tar.extractall(LOCAL_ROOT)
else:
    raise FileNotFoundError(
        f"Could not find 'cifar-10-batches-py' or 'cifar-10-python.tar.gz' in {DRIVE_DATA_DIR}. "
        "Check DRIVE_DATA_DIR in the first code cell."
    )

transform = transforms.ToTensor()

# download=False: we only use the data from your Drive
train_data = datasets.CIFAR10(root=LOCAL_ROOT, train=True,  download=False, transform=transform)
test_data  = datasets.CIFAR10(root=LOCAL_ROOT, train=False, download=False, transform=transform)
print("Loaded CIFAR-10:", len(train_data), "train /", len(test_data), "test (all 10 classes)")

## 4. Cat/Dog subset and data loaders

In [ ]:
# CIFAR-10 class IDs: cat = 3, dog = 5
class CatDogDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset
        # Use .targets so we don't decode every image just to read its label (much faster)
        self.indices = [i for i, label in enumerate(dataset.targets) if label in (3, 5)]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        image, label = self.dataset[self.indices[index]]
        label = 0 if label == 3 else 1   # cat = 0, dog = 1
        return image, label

train_dataset = CatDogDataset(train_data)
test_dataset  = CatDogDataset(test_data)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("Training images:", len(train_dataset))
print("Testing images:", len(test_dataset))

## 5. Model, loss, optimizer

In [ ]:
class CatDogCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 64),
            nn.ReLU(),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = CatDogCNN().to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

## 6. Checkpointing, resume, and plotting helpers
- `latest.pth` is written to Drive after **every epoch** (model + optimizer + history), so you can resume after a disconnect.
- It's written to a temp file first and then swapped in, so a cut-off mid-save can't corrupt your checkpoint.

In [ ]:
def save_checkpoint(path, epoch, model, optimizer, history, best_acc):
    tmp = path + ".tmp"
    torch.save({
        "epoch": epoch,                       # number of epochs completed
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "history": history,
        "best_acc": best_acc,
    }, tmp)
    os.replace(tmp, path)                     # atomic swap

def plot_history(history, save_path=None, show=True):
    epochs_range = range(1, len(history["train_loss"]) + 1)
    best_epoch = history["test_loss"].index(min(history["test_loss"])) + 1
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(epochs_range, history["train_loss"], marker="o", label="Train loss")
    ax1.plot(epochs_range, history["test_loss"],  marker="o", label="Test loss")
    ax1.axvline(best_epoch, color="red", linestyle="--", alpha=0.7,
                label=f"Lowest test loss (epoch {best_epoch})")
    ax1.set_title("Loss per epoch"); ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
    ax1.grid(alpha=0.3); ax1.legend()

    ax2.plot(epochs_range, history["train_acc"], marker="o", label="Train accuracy")
    ax2.plot(epochs_range, history["test_acc"],  marker="o", label="Test accuracy")
    ax2.axvline(best_epoch, color="red", linestyle="--", alpha=0.7,
                label=f"Overfitting starts after epoch {best_epoch}")
    ax2.set_title("Accuracy per epoch"); ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy (%)")
    ax2.grid(alpha=0.3); ax2.legend()

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150)
    if show:
        plt.show()
    plt.close(fig)

def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            total_loss += loss_function(outputs, labels).item() * labels.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, 100 * correct / total

# ---- Resume from Drive if a checkpoint exists ----
history = {"train_loss": [], "test_loss": [], "train_acc": [], "test_acc": []}
start_epoch = 0
best_acc = 0.0

if os.path.exists(LATEST_CKPT):
    ckpt = torch.load(LATEST_CKPT, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    history = ckpt["history"]
    start_epoch = ckpt["epoch"]
    best_acc = ckpt["best_acc"]
    print(f"Resumed from checkpoint: {start_epoch} epochs already done, best test acc {best_acc:.2f}%")
else:
    print("No checkpoint found - starting fresh.")

## 7. Train
Graphs update after every epoch and are saved to Drive. A red dashed line marks the epoch with the lowest test loss; training after that point is overfitting. Early stopping halts training after `PATIENCE` epochs with no improvement. If the session dies, just re-run all cells and training picks up where it left off.
To start over from scratch, delete the `checkpoints` folder in your Drive project folder.

In [ ]:
PATIENCE = 5  # stop if test loss hasn't improved for this many epochs

for epoch in range(start_epoch, EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    t0 = time.time()

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        predictions = model(images)
        loss = loss_function(predictions, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        correct += (predictions.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = 100 * correct / total
    test_loss, test_acc = evaluate(model, test_loader)

    history["train_loss"].append(train_loss)
    history["test_loss"].append(test_loss)
    history["train_acc"].append(train_acc)
    history["test_acc"].append(test_acc)

    # Best epoch = lowest test loss (derived from history, so it survives resuming)
    best_epoch = history["test_loss"].index(min(history["test_loss"])) + 1
    epochs_without_improvement = len(history["test_loss"]) - best_epoch

    if best_epoch == len(history["test_loss"]):   # this epoch is the best so far
        best_acc = test_acc
        torch.save(model.state_dict(), BEST_CKPT)

    # Save everything to Drive at the end of each epoch
    save_checkpoint(LATEST_CKPT, epoch + 1, model, optimizer, history, best_acc)
    with open(HISTORY_PATH, "w") as f:
        json.dump(history, f)

    # Redraw the graph (and save the PNG to Drive)
    clear_output(wait=True)
    plot_history(history, save_path=PLOT_PATH)
    print(
        f"Epoch {epoch + 1}/{EPOCHS} ({time.time() - t0:.1f}s) | "
        f"Train loss {train_loss:.4f}, acc {train_acc:.2f}% | "
        f"Test loss {test_loss:.4f}, acc {test_acc:.2f}% | "
        f"Best epoch: {best_epoch} | No improvement for {epochs_without_improvement}/{PATIENCE}"
    )

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping: test loss hasn't improved since epoch {best_epoch}.")
        break

print("Training finished.")

## 8. Final evaluation and save the model

In [ ]:
# Load the best weights (lowest test loss), not the possibly-overfit last epoch
model.load_state_dict(torch.load(BEST_CKPT, map_location=device))

final_loss, final_acc = evaluate(model, test_loader)
print(f"Final test accuracy (best-epoch weights): {final_acc:.2f}%")

torch.save(model.state_dict(), FINAL_MODEL)
print("Final model saved to:", FINAL_MODEL)

# Show the saved graph one more time
plot_history(history, save_path=PLOT_PATH)